In [1]:
import sys
from pathlib import Path
import torch

ROOT = Path.cwd().parent  # notebook/ 的 parent
sys.path.insert(0, str(ROOT))

import dlphys.models  # trigger registration
from dlphys.config.base import ExperimentConfig
from dlphys.config.registry import build_model
from dlphys.analysis.jvp import jvp_F
from dlphys.utils.seed import set_seed
from dlphys.analysis.lyapunov_projected import (
    lyapunov_max_benettin_projected_final,
    lyapunov_projected_per_step,
)

set_seed(0, deterministic=True)

L = 100
d_model = 16
d_k = 32
gamma = 0.5

cfg = ExperimentConfig(
    project_name="toy",
    device="cpu",
    seed=0,
    deterministic=True,
    extra={
        "model_name": "toy_attention",
        "model_kwargs": dict(d_model=d_model, d_k=d_k, L=L, num_heads=1, gamma=gamma, phi="identity"),
    }
)
m = build_model(cfg).eval()

B = 1
s = torch.randn(B, L+1, d_model)

# Attention model

In [4]:
project_slot0 = lambda v: v[:, 0, :]  # Pi_x

def F(s_in):
    # IMPORTANT: ensure this returns the next state with the SAME shape [B, L+1, d_model]
    return m(s_in)
s0 = s.clone()
out = lyapunov_projected_per_step(
    F, s0,
    T=1000, burn_in=200,
    project_fn=project_slot0,
    return_traj=False
)

print("lambda_full_mean:", out["lambda_full_mean"].item())
print("lambda_proj_mean:", out["lambda_proj_mean"].item())

lambda_full_mean: 0.0013225085567682981
lambda_proj_mean: 0.001673977356404066


In [13]:
from dlphys.analysis.lyapunov import lyapunov_max_benettin

# 用同一个初态（或拷贝一份）
s0 = s.clone()

F = lambda _s: m(_s)

out = lyapunov_max_benettin(
    F, s0,
    T=1000, burn_in=200,
    return_traj=False
)

print("lambda_hat (per batch):", out["lambda_hat"])
print("lambda_mean:", out["lambda_mean"])


@torch.no_grad()
def sensitivity_profile(m, s, n_trials=5):
    """
    Returns: sens[ L+1 ] where sens[tau] = E || d x_next / d s_tau || (via JVP)
    """
    B, T, d = s.shape
    assert B == 1
    F = lambda _s: m(_s)  # returns full s_next

    sens = torch.zeros(T)
    for tau in range(T):
        vals = []
        for _ in range(n_trials):
            v = torch.zeros_like(s)
            v[:, tau, :] = torch.randn_like(v[:, tau, :])  # perturb only this slot
            jvp = jvp_F(F, s, v)  # shape [1,T,d]
            dxnext = jvp[:, 0, :]  # output token perturbation
            vals.append(dxnext.norm(dim=-1).item())
        sens[tau] = sum(vals) / len(vals)
    return sens

sens = sensitivity_profile(m, s, n_trials=3)
print("sens shape:", sens.shape)
print("sens (first 10):", sens[:10])
print("ratio max/min:", (sens.max()/ (sens.min()+1e-12)).item())

p = sens / (sens.sum() + 1e-12)
entropy = (-(p * (p + 1e-12).log()).sum()).item()
entropy_uniform = torch.log(torch.tensor(float(L+1))).item()
print("entropy proxy:", entropy, "uniform entropy:", entropy_uniform)

lambda_hat (per batch): tensor([0.0017])
lambda_mean: tensor(0.0017)
sens shape: torch.Size([101])
sens (first 10): tensor([0.7623, 0.0323, 0.0468, 0.0400, 0.0418, 0.0213, 0.0181, 0.0504, 0.0541,
        0.0264])
ratio max/min: 85.64351654052734
entropy proxy: 4.220008850097656 uniform entropy: 4.6151204109191895
